# วิเคราะห์ความเข้าใจการคัดแยกขยะในโรงเรียน
โครงงาน **school-waste-ai** · รันได้ทันทีใน Google Colab: เมนู Runtime → Run all

ข้อมูล: แบบสอบถามขยะ 5 รายการ 305 คน ก่อนและหลังติดโปสเตอร์ (ลบอีเมลผู้ตอบแล้ว)

## 1. อ่านข้อมูลจาก Repository

In [ ]:
import pandas as pd
BASE = "https://raw.githubusercontent.com/ssukree/school-waste-ai/main/data/"
df = pd.read_csv(BASE + "survey_kaya_clean.csv")        # ก่อนติดโปสเตอร์
df_post = pd.read_csv(BASE + "survey_kaya_post.csv")    # หลังติดโปสเตอร์ 2 สัปดาห์
df.shape, df_post.shape

## 2. เทียบคำตอบกับเฉลย (ถูก = 1, ผิด = 0)

In [ ]:
key = {"ขวดน้ำ ยังไม่เปิดฝา": "เตรียมก่อน แล้วทิ้งถังรีไซเคิล",
       "ขวดมีหลอด น้ำเหลือครึ่ง": "เตรียมก่อน แล้วทิ้งถังรีไซเคิล",
       "กล่องอาหาร ล้างสะอาด": "ทิ้งถังรีไซเคิลได้ทันที",
       "กล่องอาหาร มีเศษติด": "เตรียมก่อน แล้วทิ้งถังรีไซเคิล",
       "แก้วกาแฟ เคลือบพลาสติก": "ทิ้งถังทั่วไป"}

def score(data):
    return pd.DataFrame({item: (data[item] == answer).astype(int) for item, answer in key.items()})

correct, correct_post = score(df), score(df_post)
(1 - correct.mean()).round(2)      # สัดส่วนผู้ตอบผิดรายรายการ (ก่อน)

## 3. จัดกลุ่มผู้ตอบด้วย K-means (k = 3)

In [ ]:
from sklearn.cluster import KMeans
model = KMeans(n_clusters=3, n_init=10, random_state=42)
df["cluster"] = model.fit_predict(correct)
correct.groupby(df["cluster"]).mean().round(2)   # อ่านลักษณะกลุ่ม: ถูกเกือบทุกข้อ = A · ผิดเฉพาะของเหลว/เศษอาหาร = B · ผิดเกือบทุกข้อ = C

## 4. ทดสอบแบบจำลองกับข้อมูลหลังติดโปสเตอร์
ใช้ `model.predict()` ของแบบจำลองเดิม **ไม่ fit ใหม่** เพื่อจัดกลุ่มด้วยเกณฑ์เดียวกับรอบก่อน

In [ ]:
df_post["cluster"] = model.predict(correct_post)
pd.DataFrame({"ก่อนติดโปสเตอร์": df["cluster"].value_counts(normalize=True),
              "หลังติดโปสเตอร์": df_post["cluster"].value_counts(normalize=True)}).round(2)